# Tool Use and Function Calling

Tool calling lets a model request an operation—such as searching, reading a database, or sending a message—through a structured interface controlled by your application.


## What to learn

- The tool schema describes its name, purpose, and valid arguments.
- The model proposes a call; application code validates and executes it.
- The result returns to the model so it can continue or answer.
- Permissions and confirmation belong in application code, not only in the prompt.

## Decision rule

Use narrow tools with clear names and non-overlapping purposes. Validate every argument, return useful errors, limit retries, and require approval for consequential actions.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# ALTERNATIVE FRAMEWORK EXAMPLE: OpenAI Agents SDK
# Import the dependencies used by this example.
import os

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run this example.")
else:
    # Imports stay inside the enabled branch, so the base course does not require
    # optional packages. Install them with: uv sync --extra alternatives
    from agents import Agent, Runner, function_tool

    @function_tool
    # Define the data shape and small operations before running them.
    def lookup_order(order_id: str) -> str:
        """Return the current status for one order ID."""
        return f"Order {order_id} is packed and ready to ship."

    # Configure the framework object; this line prepares it but may not execute it yet.
    agent = Agent(
        name="Order support",
        model="gpt-5.6-sol",
        instructions="Use lookup_order for order status; never invent a status.",
        tools=[lookup_order],
    )

    # Jupyter supports top-level await. Runner executes tool calls and sends their
    # results back to the model until it produces a final answer.
    result = await Runner.run(agent, "Where is order A-104?")
    print(result.final_output)


In [ ]:
"""The full tool-use loop with parallel calls and errors-as-observations.

A scripted fake model demonstrates the exact message protocol; swap
`fake_model` for a real API client and the loop is production-shaped.
"""
# Import the dependencies used by this example.
import json

TOOLS = {
    "get_invoice": {
        "description": "Look up an invoice by id. Use when the user references a specific invoice number.",
        "execute": lambda args: {"id": args["id"], "amount": 420, "status": "overdue"},
        "validate": lambda args: None if isinstance(args.get("id"), str) else "id must be a string",
    },
    "get_customer": {
        "description": "Look up a customer profile by email.",
        "execute": lambda args: (_ for _ in ()).throw(TimeoutError("upstream timeout")),
        "validate": lambda args: None,
    },
}

SCRIPT = [  # what the fake model does each turn
    {"tool_calls": [  # turn 1: two PARALLEL calls
        {"call_id": "c1", "tool": "get_invoice", "args": {"id": "INV-9"}},
        {"call_id": "c2", "tool": "get_customer", "args": {"email": "a@b.co"}},
    ]},
    {"text": "Invoice INV-9 is $420 overdue. (Customer lookup failed upstream; "
             "I proceeded with invoice data only.)"},  # turn 2: recovers from the error
]


# Define the data shape and small operations before running them.
def fake_model(messages, turn):
    return SCRIPT[turn]


def execute_call(call):
    """Validate, execute, and ALWAYS return an observation — never raise."""
    tool = TOOLS[call["tool"]]
    if (err := tool["validate"](call["args"])):
        return {"call_id": call["call_id"], "error": f"invalid args: {err}"}
    try:
        return {"call_id": call["call_id"], "result": tool["execute"](call["args"])}
    except Exception as e:
        return {"call_id": call["call_id"],
                "error": f"{type(e).__name__}: {e}", "retryable": True}


def run_agent(user_msg, max_steps=5):
    messages = [{"role": "user", "content": user_msg}]
    for turn in range(max_steps):
        reply = fake_model(messages, turn)
        if "text" in reply:
            return reply["text"]
        results = [execute_call(c) for c in reply["tool_calls"]]  # parallel in prod
        messages.append({"role": "tool_results", "content": results})
        print(f"turn {turn}: executed {len(results)} calls ->")
        for r in results:
            print("   ", json.dumps(r))
    return "step budget exhausted"


print(run_agent("Is invoice INV-9 paid? Also pull up the customer."))


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
